# 📚 Designing Data-Intensive Applications
## Bölüm 1: Trade-Offs in Data Systems Architecture
### (Veri Sistemi Mimarisinde Dengeler ve Takas-Offs)

---

> *"Çözümler yoktur; yalnızca trade-off'lar vardır. […] Ama elinden geldiğince en iyi trade-off'u bulmaya çalışırsın, ve umabileceğin tek şey de budur."*
> — Thomas Sowell, Fred Barnes ile röportaj (2005)

---

Bu bölümde, modern veri sistemlerinin mimarisinde karşılaşılan temel **trade-off**'lar (dengeler/takas) ele alınmaktadır. Hiçbir yaklaşım evrensel olarak diğerinden üstün değildir; her şeyin artıları ve eksileri vardır. Bu bölümde aşağıdaki ana başlıklar incelenmektedir:

1. **Operational vs Analytical Systems** (İşlemsel vs Analitik Sistemler)
2. **Cloud vs Self-Hosting** (Bulut vs Kendi Altyapında Barındırma)
3. **Distributed vs Single-Node Systems** (Dağıtık vs Tek Düğümlü Sistemler)
4. **Data Systems, Law, and Society** (Veri Sistemleri, Hukuk ve Toplum)

---
## 🔑 Temel Kavramlar: Data-Intensive vs Compute-Intensive

Günümüzde uygulama geliştirmenin merkezinde **data** (veri) yatmaktadır. Web ve mobil uygulamalar, **SaaS (Software as a Service)** ve **cloud services** ile birlikte, milyonlarca farklı kullanıcıdan gelen veriyi paylaşımlı, sunucu tabanlı bir **data infrastructure** (veri altyapısı) üzerinde depolamak artık olağan bir pratik haline gelmiştir.

Kullanıcı aktivitesi, iş işlemleri, cihazlar ve sensörlerden gelen verinin hem depolanması hem de analiz için erişilebilir kılınması gerekmektedir.

### Data-Intensive Application (Veri-Yoğun Uygulama) Nedir?

Bir uygulamayı **data-intensive** (veri-yoğun) olarak tanımlarız; eğer veri yönetimi, uygulamayı geliştirmedeki birincil zorluklardan biriyse.

**Compute-intensive** (hesaplama-yoğun) sistemlerde zorluk, büyük bir hesaplamayı paralel hale getirmektir. Oysa **data-intensive** uygulamalarda asıl endişe şunlardır:
- Büyük veri hacimlerini depolamak ve işlemek
- Verideki değişiklikleri yönetmek
- Hatalar ve eşzamanlılık (concurrency) karşısında tutarlılığı (consistency) sağlamak
- Servislerin yüksek erişilebilir (highly available) olmasını garantilemek

### Standart Building Blocks (Temel Yapı Taşları)

Data-intensive uygulamalar genellikle şu standart yapı taşlarından inşa edilir:

| Building Block | Türkçe Açıklama |
|---|---|
| **Databases** | Veriyi depolama; ileride bulunabilmesi için |
| **Caches** | Pahalı bir işlemin sonucunu hatırlama; okuma hızını artırmak için |
| **Search indexes** | Kullanıcıların veriyi anahtar kelimeyle aramasına veya filtrelemesine izin vermek |
| **Stream processing** | Olayları ve veri değişikliklerini anında işleme |
| **Batch processing** | Biriken büyük veri hacmini periyodik olarak işleme |

Uygulama inşa ederken, genellikle birkaç yazılım sistemi veya servisi (database, API gibi) alır ve bunları application code ile birleştiririz. Eğer sistemlerin tam olarak tasarlandığı şeyi yapıyorsak bu kolaydır; ancak uygulama büyüdükçe zorluklar ortaya çıkar.

---
## 🏢 Sidebar: Frontend ve Backend Terminolojisi

Bu kitabın büyük bölümü **backend development** ile ilgilidir:

- **Frontend**: Web uygulamalarında tarayıcıda çalışan client-side kod. Mobil uygulamalar da frontend'e benzer; kullanıcı arayüzü sağlarlar.
- **Backend**: Kullanıcı isteklerini işleyen server-side kod.

Frontend, yalnızca tek bir kullanıcının verisini yönetirken, **backend tüm kullanıcıların** verisini yönetir — bu yüzden en büyük **data infrastructure** zorlukları backend'de yatar.

Bir **backend service**:
- Genellikle **HTTP** (veya bazen **WebSocket**) üzerinden erişilebilirdir
- Bir veya daha fazla veritabanında veri okuyup yazan application code'dan oluşur
- **Cache** veya **message queue** gibi ek veri sistemleriyle de arayüz kurabilir — bunların tümüne **data infrastructure** diyebiliriz
- Çoğunlukla **stateless** (durumsuz) tasarlanır: bir HTTP isteğini işledikten sonra o istekle ilgili her şeyi unutur. Bir istekten diğerine devam etmesi gereken bilgilerin ya istemcide ya da server-side data infrastructure'da saklanması gerekir.

---
## 📊 1. Operational vs Analytical Systems
### (İşlemsel ve Analitik Sistemler)

Bir kurumda verilerle çalışan birkaç farklı insan tipi vardır:

### Kimlerin Veriye İhtiyacı Var?

1. **Backend engineers**: Veri okuma ve güncelleme isteklerini işleyen servisleri inşa eder. Bu servisler genellikle dış kullanıcılara doğrudan veya diğer servisler aracılığıyla hizmet verir.

2. **Business analysts**: Organizasyonun aktiviteleri hakkında raporlar üretir; yönetime daha iyi kararlar alması için **business intelligence (BI)** sağlar.

3. **Data scientists**: Veride yeni içgörüler arar ya da veri analizi ve **machine learning (ML)/AI** ile desteklenen kullanıcıya yönelik ürün özellikleri oluşturur (örn. e-ticaret sitesinde "X'i alanlar Y'yi de aldı" tavsiyeleri, spam filtreleme, risk skorlaması).

### İki Temel Sistem Türü

Bu durum, bu kitapta sürekli kullanılacak iki sistem türü ayrımına yol açar:

| Özellik | **Operational Systems** | **Analytical Systems** |
|---|---|---|
| Amaç | Verinin oluşturulduğu backend servisleri ve data infrastructure | Business analyst ve data scientist'lerin ihtiyaçlarına hizmet eder |
| Veri | Hem okunur hem değiştirilir | Salt okunur (read-only) kopya |
| Örnek | Kullanıcı işlemlerini işleyen servisler | Raporlama, ML modeli eğitimi |

### Yeni Roller

Bu sistemler olgunlaştıkça iki yeni uzmanlaşmış rol ortaya çıkmıştır:
- **Data engineers**: Operational ve analytical sistemleri entegre etmeyi bilen ve organizasyonun data infrastructure'ından daha geniş sorumluluk alan kişilerdir.
- **Analytics engineers**: Veriyi modelleyip dönüştürerek business analyst ve data scientist'ler için daha kullanışlı hale getirir.

---
### 🔄 OLTP ve OLAP: Transaction Processing vs Analytics

İş verisi işlemenin ilk günlerinde, bir database'e yazma işlemi genellikle ticari bir işleme karşılık geliyordu: bir satış yapmak, bir tedarikçiye sipariş vermek, bir çalışana maaş ödemek gibi. Database'ler para değişimi içermeyen pek çok alana genişlese de **transaction** terimi kalıcılaştı; mantıksal bir birimi oluşturan okuma ve yazma grubuna işaret eder.

### OLTP (Online Transaction Processing)

Operational sistemler tipik olarak bir key (anahtar) ile az sayıda kaydı sorgular — buna **point query** denir. Kayıtlar kullanıcı girdisine göre eklenir, güncellenir veya silinir. Bu uygulamalar interaktif olduğu için bu erişim modeli **OLTP** olarak adlandırılmıştır.

### OLAP (Online Analytical Processing)

Analytical sorgular OLTP'den çok farklı erişim desenlerine sahiptir:
- Çok sayıda kaydı tarar
- Bireysel kayıtlar yerine toplu istatistikler (count, sum, average) hesaplar

Örneğin bir süpermarket zincirindeki business analyst şu soruları sormak isteyebilir:
- Ocak ayında her bir mağazamızın toplam geliri ne kadardı?
- Son kampanyamızda normalden kaç adet daha fazla muz sattık?
- Hangi bebek maması markası en sık X marka bezlerle birlikte satın alınıyor?

### OLTP vs OLAP Karşılaştırması

| Özellik | **Operational (OLTP)** | **Analytical (OLAP)** |
|---|---|---|
| Ana okuma deseni | Point query (key ile bireysel kayıt) | Çok sayıda kayıt üzerinde aggregate |
| Ana yazma deseni | Bireysel kayıt oluşturma/güncelleme/silme | Toplu import (ETL) veya event stream |
| İnsan kullanıcı örneği | Web/mobil uygulama son kullanıcısı | Karar destek için dahili analist |
| Makine kullanım örneği | Bir işlemin yetkilendirilip yetkilendirilmediğini kontrol etme | Dolandırıcılık/kötüye kullanım kalıplarını tespit etme |
| Query türü | Uygulama tarafından önceden tanımlanmış, sabit | Analistler tarafından keyfi, ad-hoc keşif |
| Query hacmi | Çok sayıda küçük sorgu | Az sayıda, her biri karmaşık sorgu |
| Veri neyi temsil eder | Verinin en güncel hali | Zaman içinde gerçekleşen olayların geçmişi |
| Dataset boyutu | Gigabyte'tan terabyte'a | Terabyte'tan petabyte'a |

> **Not**: OLAP'taki "online" kelimesinin anlamı belirsizdir; muhtemelen sorgular sadece önceden tanımlanmış raporlar için değil, analistlerin OLAP sistemini keşif amacıyla interaktif kullanabileceğini gösterir.

OLTP sistemlerinde kullanıcıların custom SQL sorguları oluşturmasına genellikle izin verilmez; erişim hakları olmadıkları verileri okuyabilir ya da değiştirebilirler, ayrıca pahalı sorgular diğer kullanıcılar için performansı etkileyebilir. Bu nedenle OLTP sistemleri çoğunlukla uygulama koduna gömülü sabit sorgu kümelerini çalıştırır.

### Real-Time Analytics (Gerçek Zamanlı Analitik)

**Product analytics** veya **real-time analytics** olarak bilinen bir başka sistem türü de mevcuttur: analytical iş yüklerine (birçok kaydı toplayan sorgular) optimize edilmiş ama kullanıcıya yönelik ürünlere gömülü sistemler. **Pinot**, **Druid** ve **ClickHouse** bu tasarım için örnek gösterilebilir. Bu sistemler veriyi gerçek zamanlı olarak alır ve düşük gecikme süresi (low-latency) ile sorgu yanıtlarına optimize edilmiştir. Buna karşın geleneksel OLAP sistemleri veriyi toplu (batch) olarak alır ve yüksek verimli sorgu işlemeye (high-throughput) odaklanır.

 ---
### 🏭 Data Warehousing

Başlangıçta aynı database'ler hem transaction processing hem de analytical sorgular için kullanılıyordu. SQL her iki sorgu türü için de işe yarıyordu. Ancak 1980'lerin sonu ve 1990'ların başında şirketler analytics için OLTP sistemlerini kullanmayı bırakıp ayrı bir database sistemine yönelmeye başladı — bu ayrı veritabanına **data warehouse** denildi.

### Neden Data Warehouse?

Büyük bir enterprise'ın onlarca, hatta yüzlerce OLTP sistemi olabilir: müşteriye yönelik web sitesini çalıştıran sistemler, fiziksel mağazalardaki satış noktası (checkout) sistemlerini kontrol edenler, depolardaki envanteri takip edenler, araç güzergahlarını planlayanlar, tedarikçileri yönetenler, çalışanları idare edenler. Her biri karmaşık ve bakımı için bir ekip gerektirdiğinden büyük ölçüde bağımsız çalışır.

Business analyst ve data scientist'lerin bu OLTP sistemlerini doğrudan sorgulaması birkaç nedenden dolayı istenilmez:

1. **Data silos** problemi: İlgilenilen veri birden fazla operational sistemde dağınık olabilir; tek bir sorguda bu veri kümelerini birleştirmek güçtür.
2. OLTP için uygun schema ve veri düzeni analytics için daha az uygundur.
3. Analytical sorgular oldukça pahalı olabilir; OLTP database'inde çalıştırmak diğer kullanıcılar için performansı olumsuz etkiler.
4. OLTP sistemleri güvenlik veya uyumluluk nedenleriyle kullanıcıların doğrudan erişemeyeceği ayrı bir ağda bulunabilir.

### ETL (Extract-Transform-Load)

Data warehouse, analistlerin OLTP operasyonlarını etkilemeden istedikleri gibi sorgu yapabilecekleri ayrı bir database'dir.

Data warehouse, şirketteki tüm OLTP sistemlerinden verinin read-only kopyasını içerir. Veri OLTP database'lerinden **extract** edilir, analiz dostu bir şemaya **transform** edilir, temizlenir ve ardından data warehouse'a **load** edilir. Bu sürece **ETL (Extract-Transform-Load)** denir.

Bazen transform ve load adımlarının sırası değiştirilebilir (yani dönüştürme, yüklemeden sonra data warehouse'da yapılır) — buna **ELT** denir.

ETL işlemlerinin veri kaynakları bazen harici SaaS ürünleri olabilir: **CRM**, e-posta pazarlama veya kredi kartı işlem sistemleri gibi. Bu sistemlere yalnızca satıcının API'si üzerinden erişilebildiğinden veritabanı doğrudan kullanılamaz. Bu harici sistemlerdeki veriyi kendi data warehouse'unuza taşımak, SaaS API üzerinden mümkün olmayan analizler yapabilmenizi sağlar. SaaS API'leri için ETL genellikle **Fivetran**, **Singer** veya **Airbyte** gibi uzmanlaşmış data connector servisleri tarafından uygulanır.

### HTAP (Hybrid Transactional/Analytical Processing)

Bazı database sistemleri, ETL gerektirmeden OLTP ve analytics'i tek sistemde etkinleştirmeyi hedefleyen **HTAP** sunar. Ancak pek çok HTAP sistemi aslında içsel olarak ortak bir arayüzün arkasına gizlenmiş bir OLTP sistemi artı ayrı bir analytical sistemden oluşur; dolayısıyla bu iki sistem arasındaki ayrım, söz konusu sistemlerin nasıl çalıştığını anlamak için hâlâ önemlidir.

HTAP ayrıca data warehouse'ların yerini almaz. Aksine, hem büyük miktarda satırı tarayan analytical sorgular gerçekleştirmesi hem de düşük gecikmeyle bireysel kayıtları okuyup güncellemesi gereken uygulamalar için kullanışlıdır. Örneğin dolandırıcılık tespiti böyle iş yüklerini içerebilir.

### Genel Eğilim

Operational ve analytical sistemlerin ayrılması daha geniş bir eğilimin parçasıdır: iş yükleri daha fazla talep ettikçe sistemler daha uzmanlaşmış ve belirli iş yüklerine daha optimize hale gelir. Genel amaçlı sistemler küçük veri hacimlerini rahatlıkla kaldırır; ancak ölçek büyüdükçe sistemler daha fazla uzmanlaşma eğilimi gösterir.

---
### 🏞️ Data Lake: Data Warehouse'un Ötesinde

Bir **data warehouse** genellikle SQL aracılığıyla sorgulanan ilişkisel (relational) bir veri modeli kullanır. Bu model, business analyst'lerin yapması gereken sorgular için iyi çalışır; ancak data scientist'lerin ihtiyaçlarına daha az uygundur:

- **Feature engineering**: Bir ML modeli eğitmek için uygun forma dönüştürme. Tablo satırlarını ve sütunlarını **feature** adı verilen sayısal değerler vektörü/matrisine çevirmek gerekir; bu SQL ile ifade edilmesi güç özel kod gerektirir.
- **NLP (Natural Language Processing)**: Metin verisini (örn. ürün yorumları) yapılandırılmış bilgiye (örn. duygu analizi, konu çıkarımı) dönüştürmek.
- **Computer vision**: Fotoğraflardan yapılandırılmış bilgi çıkarmak.

Data scientist'lerin çoğu **Pandas**, **scikit-learn** gibi Python kütüphanelerini, **R** gibi istatistiksel analiz dillerini ve **Spark** gibi dağıtık analytics framework'lerini tercih eder.

### Data Lake Nedir?

**Data lake**: ETL süreçleri aracılığıyla operational sistemlerden alınan ve analizde kullanışlı olabilecek her türlü verinin kopyasını barındıran merkezi bir data repository'dir.

Data warehouse'dan farkı, data lake'in belirli bir dosya formatı, veri modeli veya şema dayatmamasıdır. Data lake'deki dosyalar:
- **Avro** veya **Parquet** gibi formatlarla kodlanmış database kayıtları koleksiyonları olabilir
- Metin, görseller, videolar, sensör okumaları, sparse matrices, feature vectors, genom dizileri veya herhangi bir türde veri de içerebilir

Data lake aynı zamanda **object store** (nesne depolama) gibi ucuz dosya depolama seçeneklerini kullanabildiğinden ilişkisel veri depolamadan genellikle daha ucuzdur.

### Sushi Prensibi

ETL süreçleri, **data pipeline**'lara doğru genişledi. Bazı durumlarda data lake, operational sistemlerden data warehouse'a giden yolda bir ara durak haline geldi. Data lake, veriyi operational sistemlerin ürettiği "ham" formda barındırır; ilişkisel data warehouse şemasına dönüştürme yapılmaz. Bu yaklaşımın avantajı, verinin her tüketicisinin ham veriyi kendi ihtiyaçlarına en uygun forma dönüştürebilmesidir. Buna bazen **sushi prensibi** denir: *"raw data is better"* (ham veri daha iyidir).

---
### 📈 Data Lake'in Ötesi: DataOps ve Stream Processing

Analytics pratikleri olgunlaştıkça organizasyonlar, analytical sistemlerin ve data pipeline'ların yönetimi ve operasyonuna giderek daha fazla önem vermeye başladı — bu **DataOps Manifesto**'da yakalanmaktadır.

Bu durum kısmen:
- **GDPR** (General Data Protection Regulation) ve **CCPA** (California Consumer Privacy Act) gibi düzenlemelerle yönetim, gizlilik ve uyumluluk sorunlarından kaynaklanmaktadır.

### Stream Processing'in Rolü

Analytics için veri artık yalnızca dosyalar ve ilişkisel tablolar olarak değil, aynı zamanda **event stream**'leri olarak da giderek daha fazla erişilebilir hale gelmektedir. Dosya tabanlı veri analizinde analizi periyodik olarak yeniden çalıştırabilirsiniz (örn. günlük), ancak **stream processing**, analytical sistemlerin olaylara çok daha hızlı — saniyeler mertebesinde — tepki vermesine olanak tanır. Bu, örneğin potansiyel olarak dolandırıcılık veya kötüye kullanım faaliyetlerini tespit etmek ve engellemek için değerli olabilir.

### Reverse ETL

Bazı durumlarda analytical sistemlerin çıktıları operational sistemlere sunulur — bu sürece **reverse ETL** denir. Örneğin, bir analytical sistemdeki verilerle eğitilen bir ML modeli, son kullanıcılara "X'i alanlar Y'yi de aldı" gibi tavsiyeler oluşturabilmesi için production ortamına deploy edilebilir. ML modelleri **TFX**, **Kubeflow** veya **MLflow** gibi uzmanlaşmış araçlar kullanılarak operational sistemlere deploy edilebilir.

---
### 📂 Systems of Record ve Derived Data

Operational ve analytical sistemler arasındaki ayrımla bağlantılı olarak, bu kitap **systems of record** ve **derived data systems** arasında da bir ayrım yapar. Bu terimler, bir sistem içindeki veri akışını netleştirmek için kullanışlıdır:

#### Systems of Record (Kayıt Sistemleri)
**System of record**, aynı zamanda **source of truth** (gerçeğin kaynağı) olarak da bilinir; verinin yetkili veya kanonik versiyonunu barındırır. Yeni veri geldiğinde — örneğin kullanıcı girdisi olarak — önce buraya yazılır. Her bilgi tam olarak bir kez temsil edilir (gösterim tipik olarak **normalized** edilmiştir). Başka bir sistem ile system of record arasında tutarsızlık varsa, system of record'daki değer (tanım olarak) doğru olandır.

#### Derived Data Systems (Türetilmiş Veri Sistemleri)
**Derived system**'daki (türetilmiş sistemdeki) veri, mevcut verinin başka bir sistemden alınarak dönüştürülmesi veya işlenmesinin sonucudur. Türetilmiş veriyi kaybetseniz bile orijinal kaynaktan yeniden oluşturabilirsiniz.

Klasik örnekler:
- **Cache**: Varsa önbellekten servis edilebilir; yoksa temel veritabanına fallback yapılır
- **Denormalized values** (denormalize edilmiş değerler)
- **Indexes** (indeksler)
- **Materialized views** (somutlaştırılmış görünümler)
- Dönüştürülmüş veri temsilleri
- Bir dataset üzerinde eğitilmiş modeller

### Önemli Noktalar

Teknik açıdan, **derived data** (türetilmiş veri) mevcut bilgiyi tekrarladığı için gereksizdir. Ancak bu veri, read query'lerin iyi performans göstermesi için sıkça zorunludur. Tek bir kaynaktan birden fazla dataset türetilebilir; bu da veriye farklı açılardan bakabilmenizi sağlar.

Analytical sistemler genellikle derived data systems'dır çünkü başka yerde oluşturulan verinin tüketicisidirler. Operational servisler, systems of record ile derived data systems'ın karışımını barındırabilir.

Önemli bir uyarı: Bir sistemdeki verinin başka bir sistemin verisinden türetildiği durumlarda, system of record'daki orijinal değiştiğinde türetilmiş veriyi güncellemek için bir süreç gereklidir. Ne yazık ki pek çok database, uygulamanızın her zaman yalnızca o tek database'i kullanacağı varsayımına dayanarak tasarlanmıştır ve bu tür güncellemeleri yayabilmek için birden fazla sistemi entegre etmeyi kolaylaştırmaz.

---
## ☁️ 2. Cloud vs Self-Hosting
### (Bulut Servisleri mi, Kendi Altyapın mı?)

Organizasyonların yapması gereken her şeyde ilk sorulardan biri: *in-house* (kendi bünyesinde) mi yapılmalı, yoksa **outsource** (dışarıdan) mı? Yani inşa etmeli misiniz, yoksa satın mı almalısınız?

Yaygın bir kural şudur: Organizasyonunuzun temel yetkinliği veya rekabet avantajı olan şeyler kendi bünyenizde yapılmalı; temel olmayan, rutin veya yaygın şeyler bir satıcıya bırakılmalıdır. Örneğin çoğu şirket kendi CPU'larını üretmez — yarı iletken üreticilerinden satın almak daha ucuzdur.

### Spektrum: Kim İnşa Eder, Kim Deploy Eder?

```
Bespoke In-House  →  Self-Hosted Open Source  →  IaaS VM  →  Cloud Service / SaaS
(Sıfırdan kendin   (MySQL indirip kendi       (VM üzerinde  (Tamamen dış satıcı;
 yazıp çalıştır)   sunucunda çalıştır)        self-hosted)   sadece API/web UI)
```

- **On premises** (kendi tesisinde): Kendi donanımınızda (genellikle kiralık bir datacenter rafında olsa bile)
- **IaaS (Infrastructure as a Service)**: Cloud'da VM (sanal makine) üzerinde self-hosted
- **Cloud service / SaaS**: Dış satıcı tarafından implement edilip işletilen, yalnızca web arayüzü veya API ile erişilen hizmet

---
### ⚖️ Cloud Servislerinin Artı ve Eksileri

Cloud servisi kullanmak, o yazılımın işletimini cloud provider'a outsource etmek anlamına gelir.

#### ✅ Cloud'un Avantajları

1. **Bilmediğin sistemleri kullanmak kolaylaşır**: Nasıl deploy edip işleteceğini bilmediğin bir sistemi benimsemek, onu yönetmeyi öğrenmekten daha kolay ve hızlıdır. Sadece o sistemi bakımı için personel kiralamak ve eğitmek çok pahalı olabilir.

2. **Uzmanlaşmış operasyon**: Bir sistemi çok sayıda müşteriye sağlamaktan operasyonel deneyim kazanan bir şirkete sistemi outsource etmek daha iyi hizmetle sonuçlanabilir.

3. **Değişken yüklerde maliyet avantajı**: Yükünüz zaman içinde çok değişiyorsa cloud servisler değerlidir. Yoğun dönemler için kapasite sağlayıp kaynakları çoğu zaman boşta bırakmak sistemin maliyet verimliliğini düşürür. Cloud servisleri talepteki değişikliklere göre computing kaynaklarını kolayca ölçeklendirmeyi sağlar.

4. **Analytical sistemler için özellikle değerli**: Büyük analitik sorguları hızlı çalıştırmak çok sayıda paralel computing kaynağı gerektirir; sorgu tamamlandığında kaynaklar boşta kalır. Cloud'da kullanılmayan kaynakları provider'a iade edebilirsiniz.

#### ❌ Cloud'un Dezavantajları

1. **Kontrol yoktur**: İhtiyacınız olan bir özellik eksikse, yalnızca satıcıya kibarca eklemelerini isteyebilirsiniz; kendiniz implement edemezsiniz.

2. **Downtime'ı beklemeniz gerekir**: Servis çökerse yapabileceğiniz tek şey düzelmesini beklemektir.

3. **Debug güçtür**: Sisteminizi tetikleyen bir bug veya performans sorunu olduğunda teşhis koymak zordur. Self-hosted yazılımda OS'den performance metric ve debug bilgisi alabilir, server log'larına bakabilirsiniz. Satıcı tarafından hosted bir serviste bu iç bilgilere erişiminiz genellikle yoktur.

4. **Vendor lock-in**: Servis kapanırsa, kabul edilemez derecede pahalı olursa veya satıcı ürününü istemediğiniz şekilde değiştirirse mecbur kalırsınız. Yazılımın eski versiyonunu çalıştırmak genellikle bir seçenek değildir. Birçok cloud servisi için standart API'ler yoktur — bu da geçiş maliyetini yükseltir.

5. **Jeopolitik risk**: Cloud provider başka bir ülkedeyse ve o ülke ile sizin ülkeniz arasında siyasi bir çatışma çıkarsa, uygulanan yaptırımlar nedeniyle servise erişiminizi kaybedebilirsiniz.

6. **Güvenlik ve uyumluluk**: Cloud provider'ın veriyi güvende tutacağına güvenmeniz gerekir; bu da gizlilik ve güvenlik düzenlemelerine uyum sürecini karmaşıklaştırabilir.

#### Ne Zaman Self-Hosting Daha Mantıklıdır?

Zaten ihtiyaç duyduğun sistemleri kurma ve işletme deneyimine sahipsen ve yükün oldukça öngörülebilirse (makinelerin sayısının çok dalgalanmadığı durumlarda), kendi makinelerini satın alıp yazılımı kendin çalıştırmak çoğu zaman daha ucuz olur.

Örneğin, çok gecikme-hassas uygulamalar (high-frequency trading gibi) donanım üzerinde tam kontrol gerektirir.

---
### 🏗️ Cloud Native System Architecture

Cloud, farklı bir ekonomik model sunmanın (donanım satın almak ve üzerinde çalıştırmak için lisans almak yerine bir servise abone olmak) ötesinde, veri sistemlerinin teknik düzeyde nasıl uygulandığı üzerinde derin bir etki bırakmıştır.

**Cloud native** terimi, cloud servislerinin avantajlarından yararlanmak üzere tasarlanmış bir mimariyi tanımlamak için kullanılır. Sıfırdan cloud native olarak tasarlanmış sistemlerin birkaç avantajı olduğu gösterilmiştir:
- Aynı donanımda daha iyi performans
- Hatalardan daha hızlı kurtarma
- Yüke göre computing kaynaklarını hızla ölçekleyebilme
- Daha büyük dataset'leri destekleyebilme

| Kategori | Self-Hosted | Cloud Native |
|---|---|---|
| Operational/OLTP | MySQL, PostgreSQL, MongoDB | AWS Aurora, Azure SQL DB Hyperscale, Google Cloud Spanner |
| Analytical/OLAP | Teradata, ClickHouse, Spark | Snowflake, Google BigQuery, Azure Synapse Analytics |

### Cloud Servislerinin Katmanlaşması

Self-hosted veri sistemlerinin çoğu basit sistem gereksinimleri vardır: Linux veya Windows gibi geleneksel bir OS üzerinde çalışır, verilerini filesystem'de dosyalar olarak saklar ve TCP/IP gibi standart network protocol'ları aracılığıyla iletişim kurar.

Cloud'da bu tür yazılımlar bir **IaaS** ortamında çalıştırılabilir: belirli CPU, bellek, disk ve network bant genişliği tahsisine sahip bir veya daha fazla **VM** (veya instance) kullanarak. Fiziksel makinelere kıyasla cloud instance'ları daha hızlı hazırlanabilir ve daha geniş çeşitlilikte boyutlarda gelir; ancak bunun dışında geleneksel bilgisayarlara benzer: istediğiniz yazılımı çalıştırabilirsiniz, ama kendiniz yönetmekten sorumlusunuz.

**Cloud native servislerinin temel fikri**, yalnızca OS'nizin yönettiği computing kaynaklarını kullanmak değil, daha üst düzey servisler oluşturmak için daha alt düzey cloud servislerinin üzerine inşa etmektir.

Örnek: **Object storage** servisleri (Amazon S3, Azure Blob Storage, Cloudflare R2) büyük dosyaları depolar. Geleneksel bir filesystem'den daha sınırlı API'ler sunarlar (temel dosya okuma ve yazma), ancak şu avantaja sahiptirler: altta yatan fiziksel makineleri gizlerler; servis verileri otomatik olarak çok sayıda makineye dağıtır, böylece herhangi bir makinede disk alanı tükenmesi konusunda endişelenmek zorunda kalmazsınız.

Örneğin **Snowflake**, veri depolama için S3'e dayanan cloud tabanlı bir analytical database (data warehouse)'dur. Bazı diğer servisler de Snowflake üzerine inşa edilmiştir.

### Separation of Storage and Compute (Depolama ve Hesaplamanın Ayrılması)

Geleneksel bilişimde disk depolama **durable** (dayanıklı) olarak kabul edilir: bir şey diske yazıldığında kaybolmayacağını varsayarız. Bireysel bir hard diskin arızasına karşı dayanıklılık sağlamak için genellikle **RAID** (Redundant Array of Independent Disks) kullanılır.

Cloud'da compute instance'larının (VM'lerin) yerel diskleri de olabilir; ancak cloud native sistemler bu diskleri genellikle **ephemeral cache** (geçici önbellek) gibi değerlendirir — yerel disk, ilişkili instance arızalanırsa veya yük değişikliğine uyum sağlamak için instance farklı fiziksel bir makineye taşınırsa erişilemez hale gelir.

Alternatif olarak cloud servisleri, **virtual disk storage** sunar: bir instance'dan ayrılıp farklı birine takılabilen (Amazon EBS, Azure managed disks, Google Cloud persistent disks gibi) depolama. Bu sanal disk fiziksel bir disk değil; her bloğun genellikle 4 KiB boyutunda olduğu bir disk davranışını taklit eden ayrı makinelerden oluşan bir cloud servisidir. Bu teknoloji geleneksel disk tabanlı yazılımı cloud'da çalıştırmayı mümkün kılar; ancak **block device emulation** ek yükler getirir ve her I/O işlemini bir network çağrısına dönüştürdüğü için uygulama network aksaklıklarına karşı hassas hale gelir.

Bu sorunu çözmek için cloud native servisler genellikle sanal disklerden kaçınır ve bunun yerine belirli iş yüklerine optimize edilmiş özel depolama servislerine dayanır. Geleneksel mimaride aynı bilgisayar hem depolama (disk) hem hesaplama (CPU ve RAM) için sorumluyken, cloud native sistemlerde bu iki sorumluluk **separation of storage and compute** (depolama ve hesaplamanın ayrışması) ile birbirinden ayrılmıştır.

### Multitenancy (Çok Kiracılı Mimari)

Cloud native sistemler çoğunlukla **multitenant**'tır: her müşteri için ayrı bir makine yerine, birden fazla müşteriden gelen veri ve hesaplama aynı paylaşılan donanımda aynı servis tarafından işlenir. Multitenancy daha iyi donanım kullanımı, daha kolay ölçeklenebilirlik ve cloud provider açısından daha kolay yönetim sağlayabilir; ancak bir müşterinin aktivitesinin diğer müşterilerin performansını veya güvenliğini etkilememesini sağlamak için dikkatli mühendislik gerektirir.

---
### 🔧 Cloud Çağında Operasyonlar

Geleneksel olarak, bir organizasyonun server-side data infrastructure'ını yöneten kişilere **DBA (database administrator)** veya **sysadmin (system administrator)** denirdi. Daha yakın zamanlarda birçok organizasyon, yazılım geliştirme ve operasyon rollerini hem backend servisleri hem de data infrastructure için ortak sorumluluk taşıyan ekiplere entegre etmeye çalışmaktadır; **DevOps** felsefesi bu eğilime yön vermiştir. **SRE (Site Reliability Engineer)**'lar Google'ın bu fikrin uygulamasıdır.

Operasyonun rolü:
- Servislerin kullanıcılara güvenilir şekilde sunulmasını sağlamak (altyapı yapılandırması ve uygulama deploy etme)
- Kararlı bir production ortamı sağlamak (güvenilirliği etkileyebilecek sorunları izleme ve teşhis etme)

Self-hosted sistemler için operasyon geleneksel olarak bireysel makineler düzeyinde önemli miktarda çalışma içerir: **capacity planning** (örn. disk alanını izleme ve tükenmeden önce ek disk ekleme), yeni makineler sağlama, servisleri bir makineden diğerine taşıma, işletim sistemi yamalarını yükleme.

Cloud servislerinin bireysel makineleri gizleyen bir API sunmasıyla birlikte bu değişmiştir. Örneğin cloud storage, sabit boyutlu disklerin yerini **metered billing** (kullandığın kadar öde) ile değiştirmiştir; böylece kapasite ihtiyaçlarını önceden planlamadan veri saklayabilir, kullandığın alan kadar ücretlendirilirsiniz.

**DevOps/SRE felsefesi** şunlara daha fazla önem verir:
- Manuel tek seferlik işler yerine tekrarlanabilir süreçler tercih ederek otomasyon kurma
- Uzun süre çalışan sunucular yerine **ephemeral VM**'ler ve servisler kullanma
- Sık uygulama güncellemelerini etkinleştirme
- Olaylardan öğrenme
- Bireyler gelip gitse bile organizasyonun sistem hakkındaki bilgisini koruma

Cloud servisleri müşterilerinin infrastructure üzerinde mümkün olduğunca az zaman ve çaba harcamasını sağlarken, altyapı şirketlerindeki operasyon ekipleri büyük müşteri kitlesine güvenilir servis sunmanın ayrıntılarında uzmanlaşır.

Cloud servislerinin müşterileri için **capacity planning finansal planlamaya**, **performance optimization maliyet optimizasyonuna** dönüşür.

---
## 🌐 3. Distributed vs Single-Node Systems
### (Dağıtık vs Tek Düğümlü Sistemler)

Bir **distributed system** (dağıtık sistem), ağ üzerinden haberleşen birden fazla makineyi içeren bir sistemdir. Bu sisteme katılan her süreç bir **node** (düğüm) olarak adlandırılır.

### Neden Distributed System Kullanılır?

| Neden | Açıklama |
|---|---|
| **Inherent distribution** | Birden fazla kullanıcı kendi cihazını kullanıyorsa sistem kaçınılmaz olarak dağıtıktır |
| **Cloud service requests** | Veri bir serviste depolanıp başka bir yerde işleniyorsa network üzerinden aktarılması gerekir |
| **Fault tolerance / High availability** | Bir makine kapansa bile başkası devreye girebilsin diye yedeklilik (redundancy) |
| **Scalability** | Veri hacmi veya computing gereksinimleri tek makinenin üstesinden gelemeyeceği kadar büyüdüğünde |
| **Latency** | Kullanıcılara yakın coğrafi konumlarda sunucu bulundurmak |
| **Elasticity** | Yoğun dönemlerde scale up, sakin dönemlerde scale down; cloud'da yalnızca aktif kullanılan kaynaklar için ödeme |
| **Specialized hardware** | ML için GPU'lu makineler, depolama için çok diskli makineler, analiz için çok CPU'lu makineler |
| **Legal compliance** | Veri yerelleştirme yasaları (data residency laws) verinin belirli bir ülke sınırları içinde tutulmasını zorunlu kılar |
| **Sustainability** | Yenilenebilir enerji fazlasının olduğu yer ve zamanda iş yüklerini çalıştırmak; karbon emisyonunu azaltmak |

---
### ⚠️ Distributed System'lerin Sorunları

Distributed system'lerin dezavantajları da vardır:

1. **Ağ hataları**: Network üzerinden giden her istek ve API çağrısı başarısızlık ihtimaliyle başa çıkmak zorundadır. Network kesintiye uğrayabilir ya da servis aşırı yüklenebilir veya çökebilir; bu nedenle herhangi bir istek yanıt alınmadan **timeout** verebilir. Bu durumda servisin isteği alıp almadığını bilmeyiz ve yalnızca yeniden denemek güvenli olmayabilir.

2. **Hız**: Datacenter network'leri hızlı olsa da başka bir servise çağrı yapmak, aynı process içinde bir fonksiyon çağırmaktan hâlâ çok daha yavaştır. Büyük veri hacimlerinde, veriyi işlemek için ayrı bir makineye transfer etmek yerine hesaplamayı verinin bulunduğu makineye taşımak daha hızlı olabilir. Bazen 100'den fazla CPU çekirdeğine sahip bir cluster'dan, tek bir threaded program çok daha iyi performans gösterebilir.

3. **Troubleshooting güçlüğü**: Sistem yavaş yanıt veriyorsa sorunun nerede olduğunu bulmak zordur. **Observability** (gözlemlenebilirlik), dağıtık sistemlerdeki sorunları teşhis etmek için kullanılan teknikleri kapsar: sistem çalışmasıyla ilgili veri toplamak ve hem üst düzey metriklerin hem de bireysel olayların analiz edilmesini sağlayan şekillerde sorgulanabilmesini sağlamak. **OpenTelemetry**, **Zipkin** ve **Jaeger** gibi **tracing** araçları, hangi client'ın hangi işlem için hangi server'ı çağırdığını ve her çağrının ne kadar sürdüğünü takip etmenizi sağlar.

4. **Data consistency**: Her servisin kendi database'i olduğunda, bu farklı servisler arasındaki veri tutarlılığını sağlamak uygulamanın sorunu haline gelir. **Distributed transactions** olası bir teknik olsa da servisler arası bağımsızlık hedefine ters düştüğü ve pek çok database bunları desteklemediği için microservices bağlamında nadiren kullanılır.

### Ne Zaman Single Node Daha İyi?

Tüm bu nedenlerle, bir görevi tek bir makinede gerçekleştirmek, distributed system kurmaktan çoğu zaman çok daha basit ve ucuzdur. CPU'lar, bellek ve diskler daha büyük, daha hızlı ve daha güvenilir hale gelmiştir. **DuckDB**, **SQLite** ve **KùzuDB** gibi single-node database'lerle birleştirildiğinde pek çok iş yükü tek bir node üzerinde çalışabilir.

---
### 🎯 Microservices ve Serverless

Bir sistemi birden fazla makineye dağıtmanın en yaygın yolu, sistemi **client**'lara ve **server**'lara bölmek ve client'ların server'lara istek yapmasına izin vermektir. Bu iletişim için en sık HTTP kullanılır.

Bu uygulama inşa biçimine geleneksel olarak **SOA (Service-Oriented Architecture)** denilmiştir; daha yakın zamanlarda ise **microservices architecture** olarak rafine edilmiştir.

### Microservices Architecture Nedir?

Bir microservices mimarisinde:
- Her servis **tek, iyi tanımlanmış bir amaca** sahiptir (örn. S3'ün amacı dosya depolama)
- Her servis, network üzerinden client'lar tarafından çağrılabilecek bir **API** sunar
- Her servisin bakımından sorumlu **tek bir ekip** vardır
- Karmaşık bir uygulama, her biri ayrı bir ekip tarafından yönetilen birden fazla birbirleriyle etkileşen servise ayrıştırılabilir

### Microservices'in Avantajları

- Her servis bağımsız olarak güncellenebilir; ekipler arası koordinasyon çabası azalır
- Her servise ihtiyaç duyduğu donanım kaynakları atanabilir
- API'nin arkasındaki implementation ayrıntılarını gizlemek, servis sahiplerinin client'ları etkilemeden implementation'ı değiştirme özgürlüğü verir
- **Veri depolama açısından**: Her servisin kendi database'ine sahip olması ve database'lerin servisler arasında paylaşılmaması yaygındır. Database paylaşmak, database yapısının tamamını servisin API'sinin bir parçası haline getirir — değiştirmesi güç olur. Paylaşılan database'ler aynı zamanda bir servisin sorgularının diğer servislerin performansını olumsuz etkilemesine de yol açabilir.

### Microservices'in Dezavantajları

- **Testing zorluğu**: Geliştirme sırasında bir servisi test etmek karmaşık olabilir; bağımlı olduğu diğer servisleri de çalıştırmak gerekir.
- **Infrastructure yükü**: Her servis için yeni sürümleri deploy etme, ayrılan donanım kaynaklarını yüke göre ayarlama, log toplama, servis sağlığını izleme ve sorun durumunda nöbetçi mühendisi uyarma için altyapı gereklidir. **Kubernetes** bu altyapı için temel sağladığı için servisleri deploy etmenin popüler bir yolu haline gelmiştir.
- **API evrimleri**: Client'lar bir API'nin belirli field'lara sahip olmasını bekler. Geliştiriciler iş gereksinimleri değiştikçe field ekleyip çıkarmak isteyebilir; ancak bu client'ların başarısız olmasına neden olabilir. **OpenAPI** ve **gRPC** gibi API açıklama standartları client-server API ilişkisini yönetmeye yardımcı olur.

Microservice'ler öncelikle **insanlar sorununa teknik bir çözümdür**: farklı ekiplerin birbirleriyle koordinasyon sağlamak zorunda kalmadan bağımsız ilerleme yapmasına izin verir. Büyük bir şirkette değerlidir; daha az ekibe sahip küçük bir şirkette ise muhtemelen gereksiz ek yük olacaktır.

### Serverless (FaaS - Function as a Service)

**Serverless** veya **FaaS**, infrastructure yönetiminin bir cloud satıcısına outsource edildiği servis deploy etmenin başka bir yaklaşımıdır.

VM kullanırken ne zaman bir instance başlatacağınızı veya kapatacağınızı açıkça seçmek zorunda kalırsınız. Buna karşın serverless modelde cloud provider, gelen isteklere göre donanım kaynaklarını otomatik olarak tahsis edip serbest bırakır.

Tıpkı cloud storage'ın sabit boyutlu disklerin yerini **metered billing** ile değiştirmesi gibi, serverless yaklaşımı da code execution için metered billing getirmektedir: kaynakları önceden provision etmek zorunda kalmak yerine yalnızca uygulama kodunuzun çalıştığı süre için ödeme yaparsınız.

Bu faydaları sunmak için pek çok serverless infrastructure provider, fonksiyon çalışma süresi üzerinde bir zaman sınırı uygular ve runtime ortamlarını kısıtlar. "Serverless" terimi de yanıltıcı olabilir; her serverless function çalışması yine de bir sunucuda çalışır, ancak sonraki çalışmalar farklı bir sunucuda gerçekleşebilir.

---
### 🖥️ Cloud Computing vs Supercomputing

Cloud computing, büyük ölçekli computing sistemleri inşa etmenin tek yolu değildir; bir alternatif **HPC (High-Performance Computing)**, yani **supercomputing**'dir.

| Özellik | **Cloud Computing** | **HPC / Supercomputing** |
|---|---|---|
| Kullanım amacı | Online servisler, iş verisi sistemleri, yüksek kullanılabilirlikte kullanıcı isteklerini karşılama | Hava tahmini, iklim modellemesi, moleküler dinamik simülasyonu, kısmi diferansiyel denklemler gibi hesaplama-yoğun bilimsel görevler |
| Hata yönetimi | Hizmet kesintisi olmaksızın devam etme | Tüm cluster iş yükünü durdur, hatalı node'u onar, son checkpoint'ten yeniden başlat |
| Network | IP ve Ethernet, Clos topolojileri, yüksek bisection bandwidth | RDMA, çok boyutlu mesh ve torus gibi özel network topolojileri |
| Güven | Karşılıklı güvensiz organizasyonlar → daha güçlü güvenlik mekanizmaları gerektirir | Sistem kullanıcıları arasında yüksek güven varsayımı |
| Coğrafi dağılım | Birden fazla coğrafi bölgeye yayılabilir | Tüm node'ların yakın olduğunu varsayar |

Büyük ölçekli analytical sistemler bazen supercomputing ile bazı özellikleri paylaşır.

---
## ⚖️ 4. Data Systems, Law, and Society
### (Veri Sistemleri, Hukuk ve Toplum)

Veri sistemlerinin mimarisi yalnızca teknik hedefler ve gereksinimlerle değil, destekledikleri organizasyonların insani ihtiyaçlarıyla da şekillenir. Veri sistemi mühendisleri giderek daha fazla, yalnızca kendi işletmelerinin ihtiyaçlarına hizmet etmenin yeterli olmadığını; aynı zamanda topluma karşı da bir sorumluluk taşıdıklarını fark etmektedir.

### Gizlilik Düzenlemeleri

İnsanlar ve davranışları hakkında veri depolayan sistemler özellikle dikkat gerektirir:

- **GDPR (General Data Protection Regulation)**: 2018'den bu yana Avrupa'daki pek çok ülkenin sakinlerine kişisel verileri üzerinde daha fazla kontrol ve yasal haklar tanır.
- **CCPA (California Consumer Privacy Act)**: Benzer gizlilik düzenlemesi.
- **EU AI Act**: AI ve kişisel verinin kullanımına ilişkin ek kısıtlamalar getirir.

### Geniş Toplumsal Etkiler

Doğrudan düzenlemeye tabi olmayan alanlarda bile bilgisayar sistemlerinin insanlar ve toplum üzerindeki etkileri giderek daha fazla kabul görmektedir:
- Sosyal medya bireylerin haberleri tüketme biçimini değiştirdi; bu da siyasi görüşleri etkiliyor
- Otomatik sistemler artık kimin kredi veya sigorta alacağını, kimin iş görüşmesine davet edileceğini, kimin suç şüphelisi olacağını belirleyen derin sonuçları olan kararlar veriyor

Bu sistemler üzerinde çalışan herkes, kararlarının etik etkisini düşünme ve ilgili yasalara uyum konusunda ortak sorumluluk taşır.

### Hukuki Gereksinim → Teknik Zorluk

Hukuki düşünceler veri sistemi tasarımının temellerini etkiliyor:

Örneğin, **GDPR bireylere talep üzerine verilerinin silinmesi hakkı** (bazen *right to be forgotten* - unutulma hakkı olarak bilinir) tanır. Ancak kitapta görüleceği gibi pek çok veri sistemi, **append-only log**'lar gibi değiştirilemez (immutable) yapılara dayanır. Değiştirilemez olması gereken bir dosyanın ortasındaki bazı verilerin silinmesi nasıl sağlanır? **Derived dataset**'lere (türetilmiş veri kümelerine) dahil edilmiş, örneğin ML modelleri için kullanılan eğitim verisindeki silmeleri nasıl ele alırız? Bu soruları cevaplamak yeni mühendislik zorlukları yaratır.

### Veri Minimizasyonu (Datensparsamkeit)

Genellikle veriyi değerinin depolama maliyetinden fazla olduğunu düşündüğümüz için depolarız. Ancak depolama maliyetlerinin S3 veya başka bir servis için ödenen fatura ile sınırlı olmadığını hatırlamak gerekir. Maliyet-fayda hesabı aynı zamanda şunları da dikkate almalıdır:
- Verinin rakipler tarafından sızdırılması veya ele geçirilmesi durumunda sorumluluk ve itibar zararı riski
- Verinin depolanması ve işlenmesinin yasalara uygun bulunmaması halinde hukuki masraflar ve para cezası riski
- Hükümetler veya kolluk kuvvetleri şirketleri veri teslim etmeye zorlayabilir. Kriminalize edilmiş davranışları (bazı Orta Doğu ve Afrika ülkelerinde eşcinsellik, bazı ABD eyaletlerinde kürtaj arayışı gibi) ortaya çıkarabilecek durumlarda bu veriyi saklamak kullanıcılar için gerçek güvenlik riskleri yaratır.

Tüm riskler hesaba katıldığında bazı verilerin depolanmaya değmeyeceğine ve silinmesi gerektiğine karar vermek makul olabilir. **Datensparsamkeit** (veri minimizasyonu) ilkesi, verinin ileride yararlı olabileceği ihtimaline karşı spekülatif olarak saklanması gereken **big data** felsefesiyle çelişir. Ancak veri minimizasyonu GDPR ile uyumludur: GDPR, kişisel verinin yalnızca belirlenmiş ve açık bir amaç için toplanabileceğini, daha sonra başka bir amaçla kullanılamayacağını ve toplandığı amaçlar için gerekenden daha uzun süre saklanamayacağını zorunlu kılar.

### Uyumluluk Standartları

- **PCI (Payment Card Industry)** standartları: Kredi kartı şirketleri, ödeme işleme işletmelerinden sıkı standartlara uymasını talep eder.
- **SOC Type 2 (Service Organization Control)**: Yazılım satıcılarına yönelik uyumluluk denetimi. Satıcılar uyumu doğrulamak için üçüncü taraf denetimlerinden geçer.

### Denge

Genel olarak işletmenizin ihtiyaçlarını, verisini topladığınız ve işlediğiniz kişilerin ihtiyaçlarıyla dengelemek önemlidir.

---
## 📋 Bölüm Özeti

Bu bölümün ana teması **trade-off**'ları anlamaktır; yani pek çok sorunun tek doğru cevabı olmadığını, her birinin artı ve eksileri olan birkaç olasılık bulunduğunu fark etmektir.

### Öğrendiklerimiz:

1. **OLTP vs OLAP**: Operational (transaction processing) ve analytical sistemler arasındaki ayrım; farklı veri türlerini farklı erişim desenleriyle yönetmekle kalmayıp aynı zamanda farklı kitlelere de hizmet eder. **Data warehouse** ve **data lake** kavramlarıyla karşılaştık; ETL süreçleri aracılığıyla operational sistemlerden veri beslemesi alırlar.

2. **Cloud vs Self-Hosting**: Cloud servisler, veri sistemi mimarisi üzerinde büyük değişiklikler getiriyor — örneğin **storage ve compute'u ayırma** biçiminde. Hangisinin daha uygun maliyetli olduğu çok fazla özel duruma bağlıdır.

3. **Distributed vs Single-Node**: Cloud sistemleri özünde dağıtıktır; ancak mümkünse sistemi tek makinede tutmak tavsiye edilir. Distributed system'lere aceleyle geçmemek iyi bir pratiktir.

4. **Data, Law, and Society**: Bir veri sisteminin mimarisi yalnızca sistemi deploy eden işletmenin ihtiyaçları tarafından değil, aynı zamanda işlenen verinin ait olduğu kişilerin haklarını koruyan gizlilik düzenlemeleri tarafından da belirlenir.

---

### 🔑 Temel Terimler Sözlüğü

| Terim | Türkçe Karşılığı / Açıklama |
|---|---|
| **Trade-off** | Denge/Takas: Bir şeyin avantajı için başka bir şeyin feda edilmesi |
| **Data-intensive** | Veri yoğun: Veri yönetiminin birincil zorluk olduğu uygulama |
| **Compute-intensive** | Hesaplama yoğun: Büyük hesaplamaları paralelleştirmenin zorluk olduğu sistem |
| **OLTP** | Online Transaction Processing: Bireysel kayıt okuma/yazma |
| **OLAP** | Online Analytical Processing: Toplu sorgu ve aggregate hesaplama |
| **ETL** | Extract-Transform-Load: Veriyi kaynaktan çekme, dönüştürme, yükleme |
| **Data warehouse** | Analitik için ayrılmış read-only veri deposu |
| **Data lake** | Ham verileri depolayan esnek merkezi depo |
| **System of record** | Verinin yetkili kaynağı |
| **Derived data** | Başka bir kaynaktan dönüştürülerek üretilen veri |
| **Cloud native** | Cloud servislerinden tam anlamıyla yararlanmak için tasarlanmış mimari |
| **IaaS** | Infrastructure as a Service: VM'ler üzerinden altyapı hizmeti |
| **SaaS** | Software as a Service: Yazılımın hizmet olarak sunulması |
| **Multitenancy** | Birden fazla müşterinin aynı donanımı paylaşması |
| **Separation of storage and compute** | Depolama ve hesaplamanın ayrılması |
| **Microservices** | Her biri tek amaçlı, bağımsız deploy edilebilir küçük servisler |
| **Serverless / FaaS** | Infrastructure yönetimini cloud provider'a bırakma |
| **DevOps** | Geliştirme ve operasyonun entegre edilmesi felsefesi |
| **SRE** | Site Reliability Engineer: Google'ın DevOps uygulaması |
| **GDPR** | General Data Protection Regulation: AB kişisel veri koruma yönetmeliği |
| **Datensparsamkeit** | Veri minimizasyonu: Yalnızca gerekli veriyi saklama prensibi |
| **Observability** | Dağıtık sistemlerin davranışını anlama/izleme kabiliyeti |
| **Vendor lock-in** | Belirli bir satıcıya aşırı bağımlılık riski |